# Create fact_sales

In [0]:
%sql
CREATE OR REPLACE TABLE bike_data.gold.fact_sales (
    sales_line_sk       BIGINT GENERATED ALWAYS AS IDENTITY,
    order_date         DATE NOT NULL,
    ship_date          DATE,
    due_date            DATE,
    customer_sk         INT NOT NULL,
    product_sk          INT NOT NULL,
    order_number        STRING NOT NULL,
    quantity            INT NOT NULL,
    unit_price          DECIMAL(10,2) NOT NULL,
    sales_amount        DECIMAL(10,2) NOT NULL,
    total_cost          DECIMAL(18,2),
    profit              DECIMAL(18,2),
    load_date           DATE NOT NULL
) USING DELTA;

# Load up fact_sales

In [0]:
query = """
WITH fixed_sales AS (
    SELECT 
        *,
        -- Fill in missing dates using other rows with the same order number
        COALESCE(order_date, MAX(order_date) OVER(PARTITION BY order_number)) as clean_order_date
    FROM bike_data.silver.crm_sales
)
SELECT
    -- Resolve Date Keys from dim_date
    d_ord.full_date     AS order_date,
    d_shp.full_date     AS ship_date,
    d_due.full_date     AS due_date,
    
    -- Resolve Dimension SKs
    c.customer_key      AS customer_sk,
    p.product_sk        AS product_sk,
    
    -- Degenerate Dimension
    s.order_number,
    
    -- Measures
    s.quantity,
    CAST(COALESCE(s.price, 0) AS DECIMAL(10,2))             AS unit_price,
    CAST(COALESCE(s.sales_amount, 0) AS DECIMAL(10,2))      AS sales_amount,
    
    -- Derived Metrics (using cost from dim_product)
    CAST(s.quantity * COALESCE(p.cost, 0) AS DECIMAL(18,2)) AS total_cost,
    CAST(s.sales_amount - (s.quantity * COALESCE(p.cost, 0)) AS DECIMAL(18,2)) AS profit,
    
    CURRENT_DATE()      AS load_date

FROM fixed_sales s
-- Join to Dim Date (3 times for different roles)
LEFT JOIN bike_data.gold.dim_date d_ord ON s.clean_order_date = d_ord.full_date
LEFT JOIN bike_data.gold.dim_date d_shp ON s.ship_date = d_shp.full_date
LEFT JOIN bike_data.gold.dim_date d_due ON s.due_date = d_due.full_date

-- Join to Dim Customer (using natural key customer_id)
INNER JOIN bike_data.gold.dim_customer c ON s.customer_id = c.customer_id

-- Join to Dim Product (using natural key product_number)
LEFT JOIN bike_data.gold.dim_products p ON s.product_number = substring(p.product_nk,7)
WHERE s.price IS NOT NULL 
    AND s.clean_order_date IS NOT NULL -- to catches any order that has order date at all
    AND s.sales_amount IS NOT NULL
"""

# Execute SQL and get DataFrame
df_fact = spark.sql(query)
df_fact.display()

In [0]:
df_fact.printSchema()

# Write to Gold table

In [0]:
# Write to the Gold Table
(
    df_fact.write
    .mode("overwrite")
    .saveAsTable("bike_data.gold.fact_sales")
)

In [0]:
df_fact.limit(7).display()